<!-- Cell 01: Notebook overview -->
# Centralized Baseline — Optimized V2

This version is optimized for fast iteration on Kaggle.

- `dataset_frac` controls the approximate fraction of raw CSV shards parsed.
- For `dataset_frac < 1`, the notebook **does not read all 63 CSV files**.
- Stable label IDs are reused from the verified full baseline.
- Pandas is removed from the training loop.
- Scaling happens once before training.
- Training data is kept on GPU when it safely fits.
- Checkpoints are saved after completed epochs.


<!-- Cell 02: CPU-GPU bottleneck fix -->
## CPU–GPU Bottleneck Fix

**Problem in the original `centralized_baseline` notebook:** the GPU handled model training, but every batch first depended on CPU-heavy Pandas/NumPy preprocessing (`df.iloc`, conversion, and `StandardScaler.transform`). The GPU therefore spent significant time waiting for the CPU/data pipeline.

**Modification:** preprocessing is now performed once before training: cleaned features are converted to contiguous `float32` arrays, scaling is applied once, tensors are created once, and training batches are served directly from tensors. When memory allows, the training tensors stay on the GPU; otherwise efficient CPU→GPU batch transfer is used. This removes repeated Pandas/scaler work from the training loop and keeps the GPU fed more consistently.

> `dataset_frac` is only a temporary experiment-control mechanism for faster iteration; it is not the core bottleneck solution.


In [ ]:
# Cell 03: Imports
from pathlib import Path
from contextlib import nullcontext
import os
import gc
import json
import time
import random
import platform

import numpy as np
import pandas as pd
import psutil
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    matthews_corrcoef,
    cohen_kappa_score,
)

import torch
import torch.nn as nn


In [ ]:
# Cell 04: Configuration
EXPERIMENT_NAME = "centralized_baseline_optimized_v2"

# Main experiment control: 0 < dataset_frac <= 1
# IMPORTANT: for frac < 1, only a deterministic subset of CSV files is parsed.
# This avoids reading/cleaning 100% of the raw dataset for a small experiment.
dataset_frac = 1.00

if not (0 < dataset_frac <= 1):
    raise ValueError("dataset_frac must satisfy 0 < dataset_frac <= 1")

DATA_DIR_CANDIDATES = [
    Path("/kaggle/input/ciot2023-2"),
    Path("/kaggle/input/datasets/mashrurfaiyaz28/ciot2023-2"),
]
DATA_DIR = next((p for p in DATA_DIR_CANDIDATES if p.exists()), DATA_DIR_CANDIDATES[0])

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
TEST_SIZE = 0.20
BATCH_SIZE = 8192
EVAL_BATCH_SIZE = 32768
EPOCHS = 3
LEARNING_RATE = 0.001
HIDDEN_1 = 128
HIDDEN_2 = 64

USE_CLASS_WEIGHTS = False
USE_AMP = True
KEEP_TRAIN_ON_GPU = True
GPU_DATA_MAX_FRACTION = 0.70
RESUME_FROM_CHECKPOINT = True

FRAC_TAG = f"frac_{dataset_frac:.3f}".replace(".", "p")
CHECKPOINT_PATH = OUTPUT_DIR / f"{EXPERIMENT_NAME}_{FRAC_TAG}_checkpoint.pt"

print("Experiment:", EXPERIMENT_NAME)
print("dataset_frac:", dataset_frac)
print("Data directory:", DATA_DIR)
print("Checkpoint:", CHECKPOINT_PATH)


In [ ]:
# Cell 05: Reproducibility and device
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def show_ram(prefix="RAM"):
    process_gb = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)
    system = psutil.virtual_memory()
    print(f"{prefix} | Process: {process_gb:.2f} GB | System used: {system.percent:.1f}%")


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"GPU memory: {free_b/1024**3:.2f} GB free / {total_b/1024**3:.2f} GB total")

show_ram("Startup")


In [ ]:
# Cell 06: Discover files, choose fraction, and build label map
all_files = sorted(DATA_DIR.rglob("Merged*.csv"))
if not all_files:
    all_files = sorted(DATA_DIR.rglob("*.csv"))
if not all_files:
    raise FileNotFoundError(f"No CSV files found under: {DATA_DIR}")

# Read only one header; do not scan all files just to discover labels.
header = pd.read_csv(all_files[0], nrows=0)
if "Label" not in header.columns:
    raise ValueError("Expected a 'Label' column.")

feature_cols = [c for c in header.columns if c != "Label"]
dtype_map = {c: np.float32 for c in feature_cols}

# Stable class order taken from the verified full centralized-baseline run.
classes = ['BACKDOOR_MALWARE', 'BENIGN', 'BROWSERHIJACKING', 'COMMANDINJECTION', 'DDOS-ACK_FRAGMENTATION', 'DDOS-HTTP_FLOOD', 'DDOS-ICMP_FLOOD', 'DDOS-ICMP_FRAGMENTATION', 'DDOS-PSHACK_FLOOD', 'DDOS-RSTFINFLOOD', 'DDOS-SLOWLORIS', 'DDOS-SYNONYMOUSIP_FLOOD', 'DDOS-SYN_FLOOD', 'DDOS-TCP_FLOOD', 'DDOS-UDP_FLOOD', 'DDOS-UDP_FRAGMENTATION', 'DICTIONARYBRUTEFORCE', 'DNS_SPOOFING', 'DOS-HTTP_FLOOD', 'DOS-SYN_FLOOD', 'DOS-TCP_FLOOD', 'DOS-UDP_FLOOD', 'MIRAI-GREETH_FLOOD', 'MIRAI-GREIP_FLOOD', 'MIRAI-UDPPLAIN', 'MITM-ARPSPOOFING', 'RECON-HOSTDISCOVERY', 'RECON-OSSCAN', 'RECON-PINGSWEEP', 'RECON-PORTSCAN', 'SQLINJECTION', 'UPLOADING_ATTACK', 'VULNERABILITYSCAN', 'XSS']
label_map = {label: i for i, label in enumerate(classes)}
id_to_label = {i: label for label, i in label_map.items()}

# TRUE fraction control for fast experiments:
# choose only approximately dataset_frac of the CSV shards BEFORE CSV parsing.
rng = np.random.default_rng(SEED)
if dataset_frac >= 1.0:
    files = all_files
else:
    n_files = max(1, int(round(len(all_files) * dataset_frac)))
    chosen_idx = np.sort(rng.choice(len(all_files), size=n_files, replace=False))
    files = [all_files[i] for i in chosen_idx]

effective_file_frac = len(files) / len(all_files)
print("Total CSV files:", len(all_files))
print("Selected CSV files:", len(files))
print("Approx. raw fraction:", f"{effective_file_frac:.3f}")
print("Features:", len(feature_cols))
print("Classes:", len(classes))
print("Selected files:", [f.name for f in files])


In [ ]:
# Cell 07: Load and clean selected CSV shards
load_start = time.perf_counter()
X_parts, y_parts = [], []
rows_raw_total = 0
rows_clean_total = 0

for i, file in enumerate(files, start=1):
    part = pd.read_csv(file, dtype=dtype_map)
    rows_raw_total += len(part)

    part.replace([np.inf, -np.inf], np.nan, inplace=True)
    part.dropna(inplace=True)
    rows_clean_total += len(part)

    unknown = set(part["Label"].dropna().unique()) - set(label_map)
    if unknown:
        raise ValueError(f"Unknown labels found: {sorted(unknown)}")

    y_part = part["Label"].map(label_map).to_numpy(dtype=np.int64, copy=True)
    X_part = part[feature_cols].to_numpy(dtype=np.float32, copy=True)

    X_parts.append(X_part)
    y_parts.append(y_part)

    print(
        f"[{i:02d}/{len(files):02d}] {file.name} | "
        f"raw={len(part):,} | kept={len(y_part):,}"
    )

    del part, X_part, y_part
    gc.collect()

print("\nCombining selected rows...")
X = np.concatenate(X_parts, axis=0)
y = np.concatenate(y_parts, axis=0)
del X_parts, y_parts
gc.collect()

load_seconds = time.perf_counter() - load_start

print("Raw rows read:", f"{rows_raw_total:,}")
print("Clean selected rows:", f"{len(y):,}")
print("X shape:", X.shape, "| dtype:", X.dtype)
print("Load time:", f"{load_seconds:.2f} s")
show_ram("After loading")


In [ ]:
# Cell 08: Dataset validation and class distribution
if not np.isfinite(X).all():
    raise ValueError("Feature matrix still contains non-finite values.")

counts = np.bincount(y, minlength=len(classes))
missing = [classes[i] for i, c in enumerate(counts) if c == 0]

class_distribution_df = pd.DataFrame({
    "class_id": np.arange(len(classes)),
    "class_name": classes,
    "count": counts,
    "percent": 100.0 * counts / len(y),
})

print(class_distribution_df.to_string(index=False))
print("\nMissing classes:", missing if missing else "None")

if missing:
    raise ValueError(
        "Selected CSV shards do not contain every class. "
        "Increase dataset_frac slightly or change SEED."
    )

if counts[counts > 0].min() < 2:
    raise ValueError("At least one class has fewer than 2 samples; increase dataset_frac.")

print("Smallest class:", int(counts[counts > 0].min()))
print("Largest class:", int(counts.max()))


In [ ]:
# Cell 09: Stratified train-test split
indices = np.arange(len(y), dtype=np.int32)
train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y,
)
del indices

# Materialize each split once so training never returns to Pandas.
X_train = np.ascontiguousarray(X[train_idx], dtype=np.float32)
y_train = np.ascontiguousarray(y[train_idx], dtype=np.int64)
X_test = np.ascontiguousarray(X[test_idx], dtype=np.float32)
y_test = np.ascontiguousarray(y[test_idx], dtype=np.int64)

# Free the combined arrays and split indices.
del X, y, train_idx, test_idx
gc.collect()

print("Training samples:", f"{len(y_train):,}")
print("Testing samples: ", f"{len(y_test):,}")
print(f"Train data size: {(X_train.nbytes + y_train.nbytes)/1024**3:.2f} GB")
show_ram("After split")


In [ ]:
# Cell 10: Fit scaler once and scale arrays once
scale_start = time.perf_counter()
feature_scaler = StandardScaler()
feature_scaler.fit(X_train)

# Apply StandardScaler statistics in-place to avoid a second full-size copy.
mean32 = feature_scaler.mean_.astype(np.float32)
scale32 = feature_scaler.scale_.astype(np.float32)

X_train -= mean32
X_train /= scale32
X_test -= mean32
X_test /= scale32

scaler_seconds = time.perf_counter() - scale_start

print("Scaling complete.")
print("Scaler time:", f"{scaler_seconds:.2f} s")
print("Train finite:", bool(np.isfinite(X_train).all()))
print("Test finite:", bool(np.isfinite(X_test).all()))


In [ ]:
# Cell 11: Create tensors and choose data location
# Test data stays on CPU because it is used only once after training.
X_test_t = torch.from_numpy(X_test)
y_test_t = torch.from_numpy(y_test)

train_bytes = X_train.nbytes + y_train.nbytes
move_train_to_gpu = False

if KEEP_TRAIN_ON_GPU and device.type == "cuda":
    free_bytes, _ = torch.cuda.mem_get_info()
    move_train_to_gpu = train_bytes < GPU_DATA_MAX_FRACTION * free_bytes

if move_train_to_gpu:
    print(f"Moving {train_bytes/1024**3:.2f} GB of training data to GPU once...")
    X_train_t = torch.from_numpy(X_train).to(device)
    y_train_t = torch.from_numpy(y_train).to(device)
    train_data_device = device
else:
    X_train_t = torch.from_numpy(X_train)
    y_train_t = torch.from_numpy(y_train)
    train_data_device = torch.device("cpu")

# Tensor objects now own/reference the data we need.
del X_train, y_train, X_test, y_test
gc.collect()

print("Training data location:", train_data_device)
print("Test data location: CPU")
if device.type == "cuda":
    print(f"GPU allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


In [ ]:
# Cell 12: Model definition
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_1=128, hidden_2=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_1),
            nn.ReLU(),
            nn.Linear(hidden_1, hidden_2),
            nn.ReLU(),
            nn.Linear(hidden_2, num_classes),
        )

    def forward(self, x):
        return self.network(x)


model = MLP(
    input_dim=len(feature_cols),
    num_classes=len(classes),
    hidden_1=HIDDEN_1,
    hidden_2=HIDDEN_2,
).to(device)

num_parameters = sum(p.numel() for p in model.parameters())
print(model)
print("Trainable parameters:", f"{num_parameters:,}")


In [ ]:
# Cell 13: Loss, optimizer, and mixed precision
if USE_CLASS_WEIGHTS:
    train_counts = np.bincount(y_train_t.detach().cpu().numpy(), minlength=len(classes)).astype(np.float64)
    if np.any(train_counts == 0):
        raise ValueError("A class has zero training samples.")

    class_weights_np = len(y_train_t) / (len(classes) * train_counts)
    class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
amp_enabled = bool(USE_AMP and device.type == "cuda")
grad_scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

print("Class weights:", USE_CLASS_WEIGHTS)
print("AMP enabled:", amp_enabled)
print("Optimizer: Adam | learning rate:", LEARNING_RATE)


In [ ]:
# Cell 14: Sanity check one batch
n = min(BATCH_SIZE, len(y_train_t))
idx = torch.arange(n, device=train_data_device)
X_batch = X_train_t[idx]
y_batch = y_train_t[idx]

if train_data_device.type == "cpu":
    X_batch = X_batch.to(device)
    y_batch = y_batch.to(device)

model.eval()
with torch.no_grad():
    ctx = torch.amp.autocast("cuda", dtype=torch.float16) if amp_enabled else nullcontext()
    with ctx:
        logits = model(X_batch)

print("X:", X_batch.shape, X_batch.dtype)
print("y:", y_batch.shape, y_batch.dtype)
print("Output:", logits.shape)
print("Outputs finite:", bool(torch.isfinite(logits).all()))

del X_batch, y_batch, logits, idx
if device.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# Cell 15: Training function
def train_one_epoch(epoch_number):
    model.train()
    n_samples = len(y_train_t)
    num_batches = (n_samples + BATCH_SIZE - 1) // BATCH_SIZE

    # Permutation lives beside the training data; batches are indexed in compiled PyTorch code.
    permutation = torch.randperm(n_samples, device=train_data_device)

    running_loss = 0.0
    correct = 0
    total = 0
    epoch_start = time.perf_counter()

    for batch_number, start in enumerate(range(0, n_samples, BATCH_SIZE), start=1):
        batch_idx = permutation[start:start + BATCH_SIZE]
        X_batch = X_train_t[batch_idx]
        y_batch = y_train_t[batch_idx]

        if train_data_device.type == "cpu":
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

        optimizer.zero_grad(set_to_none=True)

        ctx = torch.amp.autocast("cuda", dtype=torch.float16) if amp_enabled else nullcontext()
        with ctx:
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

        grad_scaler.scale(loss).backward()
        grad_scaler.step(optimizer)
        grad_scaler.update()

        batch_n = y_batch.size(0)
        running_loss += loss.item() * batch_n
        correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total += batch_n

        if batch_number % 250 == 0 or batch_number == num_batches:
            elapsed = time.perf_counter() - epoch_start
            print(
                f"Epoch {epoch_number}/{EPOCHS} | Batch {batch_number:,}/{num_batches:,} | "
                f"Loss {running_loss/total:.4f} | Acc {100*correct/total:.2f}% | "
                f"{total/elapsed:,.0f} samples/s"
            )

    epoch_seconds = time.perf_counter() - epoch_start
    del permutation

    return {
        "epoch": epoch_number,
        "train_loss": running_loss / total,
        "train_accuracy": correct / total,
        "epoch_seconds": epoch_seconds,
        "samples_per_second": total / epoch_seconds,
    }


In [ ]:
# Cell 16: Resume and train with epoch checkpoints
history = []
start_epoch = 1

if RESUME_FROM_CHECKPOINT and CHECKPOINT_PATH.exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if "amp_scaler_state_dict" in checkpoint:
        grad_scaler.load_state_dict(checkpoint["amp_scaler_state_dict"])
    history = checkpoint.get("history", [])
    start_epoch = int(checkpoint.get("epoch", 0)) + 1
    print(f"Resuming from completed epoch {start_epoch - 1}.")

training_start = time.perf_counter()

for epoch in range(start_epoch, EPOCHS + 1):
    metrics = train_one_epoch(epoch)
    history.append(metrics)

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "amp_scaler_state_dict": grad_scaler.state_dict(),
        "history": history,
        "dataset_frac": dataset_frac,
        "classes": classes,
        "feature_cols": feature_cols,
        "scaler_mean": feature_scaler.mean_,
        "scaler_scale": feature_scaler.scale_,
    }, CHECKPOINT_PATH)

    print(
        f"Epoch {epoch} complete | {metrics['epoch_seconds']:.2f} s | "
        f"{metrics['samples_per_second']:,.0f} samples/s | checkpoint saved"
    )

training_session_seconds = time.perf_counter() - training_start
history_df = pd.DataFrame(history)

print("\nCompleted history:")
print(history_df.to_string(index=False))
print("This session training time:", f"{training_session_seconds:.2f} s")


In [ ]:
# Cell 17: Training history plots
if not history_df.empty:
    plt.figure(figsize=(7, 4))
    plt.plot(history_df["epoch"], history_df["train_loss"], marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Training Loss")
    plt.title("Training Loss")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(history_df["epoch"], history_df["train_accuracy"], marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Training Accuracy")
    plt.title("Training Accuracy")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


In [ ]:
# Cell 18: Evaluation function
def evaluate_model():
    model.eval()
    predictions = np.empty(len(y_test_t), dtype=np.int16)
    targets = np.empty(len(y_test_t), dtype=np.int16)

    start_time = time.perf_counter()
    write_pos = 0

    with torch.no_grad():
        for start in range(0, len(y_test_t), EVAL_BATCH_SIZE):
            X_batch = X_test_t[start:start + EVAL_BATCH_SIZE].to(device)
            y_batch = y_test_t[start:start + EVAL_BATCH_SIZE].to(device)

            ctx = torch.amp.autocast("cuda", dtype=torch.float16) if amp_enabled else nullcontext()
            with ctx:
                logits = model(X_batch)

            pred = logits.argmax(dim=1)
            n = len(y_batch)
            predictions[write_pos:write_pos + n] = pred.cpu().numpy()
            targets[write_pos:write_pos + n] = y_batch.cpu().numpy()
            write_pos += n

    return targets, predictions, time.perf_counter() - start_time


In [ ]:
# Cell 19: Run evaluation and overall metrics
test_targets, test_predictions, evaluation_seconds = evaluate_model()

accuracy = accuracy_score(test_targets, test_predictions)
balanced_accuracy = balanced_accuracy_score(test_targets, test_predictions)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    test_targets, test_predictions, average="macro", zero_division=0
)
weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    test_targets, test_predictions, average="weighted", zero_division=0
)
mcc = matthews_corrcoef(test_targets, test_predictions)
cohen_kappa = cohen_kappa_score(test_targets, test_predictions)

overall_metrics = {
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_accuracy),
    "macro_precision": float(macro_precision),
    "macro_recall": float(macro_recall),
    "macro_f1": float(macro_f1),
    "weighted_precision": float(weighted_precision),
    "weighted_recall": float(weighted_recall),
    "weighted_f1": float(weighted_f1),
    "mcc": float(mcc),
    "cohen_kappa": float(cohen_kappa),
}

for name, value in overall_metrics.items():
    print(f"{name:<24} {value:.4f}")
print("Evaluation time:", f"{evaluation_seconds:.2f} s")


In [ ]:
# Cell 20: Per-class metrics and report
report_dict = classification_report(
    test_targets,
    test_predictions,
    labels=np.arange(len(classes)),
    target_names=classes,
    output_dict=True,
    zero_division=0,
)

per_class_df = pd.DataFrame([
    {
        "class_id": i,
        "class_name": name,
        "precision": report_dict[name]["precision"],
        "recall": report_dict[name]["recall"],
        "f1": report_dict[name]["f1-score"],
        "support": int(report_dict[name]["support"]),
    }
    for i, name in enumerate(classes)
])

print(per_class_df.sort_values("f1").to_string(index=False))


In [ ]:
# Cell 21: Normalized confusion matrix
cm_norm = confusion_matrix(
    test_targets,
    test_predictions,
    labels=np.arange(len(classes)),
    normalize="true",
)

plt.figure(figsize=(18, 16))
plt.imshow(cm_norm, interpolation="nearest", aspect="auto")
plt.colorbar()
plt.title(f"Centralized Baseline — Normalized Confusion Matrix ({dataset_frac:.0%} data)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(np.arange(len(classes)), classes, rotation=90, fontsize=8)
plt.yticks(np.arange(len(classes)), classes, fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 22: Comparison-ready headline metrics
comparison_metrics_df = pd.DataFrame([{
    "dataset_frac": float(dataset_frac),
    "train_samples": int(len(y_train_t)),
    "test_samples": int(len(y_test_t)),
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_accuracy),
    "macro_precision": float(macro_precision),
    "macro_recall": float(macro_recall),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "mcc": float(mcc),
    "cohen_kappa": float(cohen_kappa),
}])

display(comparison_metrics_df.round(4))


In [ ]:
# Cell 23: Per-class F1 comparison graph
plot_df = per_class_df.sort_values("f1", ascending=True)

plt.figure(figsize=(10, 11))
plt.barh(plot_df["class_name"], plot_df["f1"])
plt.xlabel("F1-score")
plt.ylabel("Class")
plt.title(f"Centralized Baseline — Per-Class F1 ({dataset_frac:.0%} data)")
plt.xlim(0, 1.0)
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 24: Runtime and throughput metrics
total_training_seconds = float(history_df["epoch_seconds"].sum()) if not history_df.empty else 0.0
mean_samples_per_second = float(history_df["samples_per_second"].mean()) if not history_df.empty else 0.0

runtime_metrics_df = pd.DataFrame([{
    "loading_s": float(load_seconds),
    "scaling_s": float(scaler_seconds),
    "training_s": total_training_seconds,
    "evaluation_s": float(evaluation_seconds),
    "mean_train_samples_per_s": mean_samples_per_second,
    "training_data_location": str(train_data_device),
    "amp_enabled": bool(amp_enabled),
}])

display(runtime_metrics_df.round(3))


In [ ]:
# Cell 25: Experiment summary
experiment_summary = {
    "experiment_name": EXPERIMENT_NAME,
    "dataset_frac_requested": float(dataset_frac),
    "dataset_file_fraction_effective": float(effective_file_frac),
    "seed": SEED,
    "dataset": {
        "raw_rows_read": int(rows_raw_total),
        "clean_selected_rows": int(len(y_train_t) + len(y_test_t)),
        "train_samples": int(len(y_train_t)),
        "test_samples": int(len(y_test_t)),
        "num_features": int(len(feature_cols)),
        "num_classes": int(len(classes)),
    },
    "model": {
        "architecture": f"{len(feature_cols)}-{HIDDEN_1}-{HIDDEN_2}-{len(classes)}",
        "trainable_parameters": int(num_parameters),
    },
    "training": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "use_amp": amp_enabled,
        "training_data_location": str(train_data_device),
        "history": history,
    },
    "metrics": overall_metrics,
    "runtime_seconds": {
        "loading": float(load_seconds),
        "scaling": float(scaler_seconds),
        "training": float(total_training_seconds),
        "evaluation": float(evaluation_seconds),
    },
    "mean_train_samples_per_second": float(mean_samples_per_second),
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "device": str(device),
        "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
    },
}

print(json.dumps(experiment_summary, indent=2))


In [ ]:
# Cell 26: Save result artifacts
results_path = OUTPUT_DIR / f"{EXPERIMENT_NAME}_{FRAC_TAG}_results.json"
per_class_path = OUTPUT_DIR / f"{EXPERIMENT_NAME}_{FRAC_TAG}_per_class.csv"
history_path = OUTPUT_DIR / f"{EXPERIMENT_NAME}_{FRAC_TAG}_history.csv"
comparison_path = OUTPUT_DIR / f"{EXPERIMENT_NAME}_{FRAC_TAG}_comparison_metrics.csv"
runtime_path = OUTPUT_DIR / f"{EXPERIMENT_NAME}_{FRAC_TAG}_runtime_metrics.csv"

with open(results_path, "w") as f:
    json.dump(experiment_summary, f, indent=4)

per_class_df.to_csv(per_class_path, index=False)
history_df.to_csv(history_path, index=False)
comparison_metrics_df.to_csv(comparison_path, index=False)
runtime_metrics_df.to_csv(runtime_path, index=False)

print("Saved:")
for path in [CHECKPOINT_PATH, results_path, per_class_path, history_path, comparison_path, runtime_path]:
    print("-", path)


<!-- Cell 27: Run guidance -->
## Run guidance

Start with `dataset_frac = 0.10`. If the throughput and results look reasonable, rerun with a larger fraction such as `0.25`, `0.50`, and finally `1.00` if needed.

Each fraction gets its own checkpoint filename, so changing `dataset_frac` will not accidentally resume a model trained on a different fraction.

In [ ]:
# Cell 28: Package all experiment outputs for download
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

output_dir = Path("/kaggle/working")
zip_path = output_dir / "centralized_baseline_all_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in output_dir.iterdir():
        if file.is_file() and file != zip_path:
            z.write(file, arcname=file.name)

print("Created:", zip_path)
display(FileLink(str(zip_path)))